In [0]:
%run "../includes/configuration"

In [0]:
movie_df = spark.read.parquet(f"{silver_folder_path}/movies").filter("year_release_date > 2010")

In [0]:
country_df = spark.read.parquet(f"{silver_folder_path}/countries")

In [0]:
movie_company_df = spark.read.parquet(f"{silver_folder_path}/movie_companies")

In [0]:
production_country_df = spark.read.parquet(f"{silver_folder_path}/production_countries")

In [0]:
production_company_df = spark.read.parquet(f"{silver_folder_path}/production_companies")

#### Join "production_countries" y "countries"

In [0]:
countries_prod_coun_df = production_country_df.join(country_df,
                                    production_country_df.country_id == country_df.country_id,
                                    "inner") \
                                    .select(country_df.country_name, production_country_df.movie_id)


#### Join "movie_companies" y "production_companies"

In [0]:
companies_prod_comp_df = movie_company_df.join(production_company_df,
                                    movie_company_df.company_id == production_company_df.company_id,
                                    "inner") \
                                    .select(production_company_df.company_name, movie_company_df.movie_id)

#### Join "movie_df", "countries_prod_coun_df" y "companies_prod_comp_df"

In [0]:
results_country_prod_company_df = movie_df.join(countries_prod_coun_df,
                                                movie_df.movie_id == countries_prod_coun_df.movie_id,
                                                "inner") \
                                            .join(companies_prod_comp_df,
                                                  movie_df.movie_id == companies_prod_comp_df.movie_id,
                                                  "inner") \
                                            .select(movie_df.title, movie_df.budget, movie_df.revenue, movie_df.duration_time, 
                                                    movie_df.release_date, countries_prod_coun_df.country_name, companies_prod_comp_df.company_name)
                                            

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
results_df = results_country_prod_company_df.withColumn("created_date", current_timestamp())


##### Ordenar por la columna "title" de manera ascendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.title.asc())

In [0]:
display(results_order_by_df)

In [0]:
results_order_by_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_country_prod_company")

In [0]:
display(spark.read.parquet(f"{gold_folder_path}/results_country_prod_company"))